# IBKR Flex sync

Pulls the "Trade History API" Flex Query (Cash Report + Open Positions + Trades) and brings `data/brokers/ibkr/` up to date. Safe to re-run: trades dedupe by transaction ID, so running this twice in a row just confirms nothing changed.

Run this regularly — the underlying Flex Query is scoped to "Last Business Day" on IBKR's side, so a missed run creates a permanent gap rather than something you can ask for later. If a run raises `TradeHistoryGapError`, see `docs/ibkr_flex_api.md` for how to backfill it. All the mechanics (protocol, error codes, why trades vs. snapshots are cached differently) are documented there too.

In [1]:
from trades.brokers import ibkr
from trades.config import IbkrFlexApiConfig, IbkrFlexCredentials

credentials = IbkrFlexCredentials()  # reads IBKR_FLEX_WEB_SERVICE_TOKEN / IBKR_QUERY_ID from .env
config = IbkrFlexApiConfig()

## Run the sync

One network round trip (SendRequest, then poll GetStatement until ready), then trades/positions/cash are all updated from that single fetched statement.

In [2]:
result = ibkr.sync_ibkr_account(credentials, config)
result

IbkrSyncResult(pulled_at=datetime.datetime(2026, 7, 1, 16, 38, 26), statement_from_date=datetime.date(2025, 7, 1), statement_to_date=datetime.date(2026, 6, 30), new_trade_count=0, total_trade_count=23)

## What's in the cache now

In [3]:
trades = ibkr.load_trade_history(config)
first_date = trades["trade_date"].min().date()
last_date = trades["trade_date"].max().date()
print(f"{len(trades)} trades cached, spanning {first_date} to {last_date}")
trades

23 trades cached, spanning 2026-01-27 to 2026-06-30


,account_id,transaction_id,trade_id,symbol,asset_category,currency,buy_sell,trade_date,quantity,trade_price,trade_money,ib_commission,net_cash
0,U24174819,37591258188,8893138032,VOO,STK,USD,BUY,2026-01-27,0.1500,640.3900,96.058500,-9.605850e-01,-97.019085
1,U24174819,39170302323,9300870154,BND,STK,USD,BUY,2026-04-10,1.6950,73.7450,124.997775,-1.000000e+00,-125.997775
2,U24174819,39170320857,9300875513,VOO,STK,USD,BUY,2026-04-10,1.3974,626.1400,874.968036,-1.000000e+00,-875.968036
3,U24174819,39170314217,9300870266,VXUS,STK,USD,BUY,2026-04-10,3.0706,81.4150,249.992899,-1.000000e+00,-250.992899
4,U24174819,39700798632,9439924745,BND,STK,USD,BUY,2026-05-05,3.0000,73.2500,219.750000,-9.000000e-06,-219.750009
5,U24174819,39700891531,9439959428,BND,STK,USD,BUY,2026-05-05,0.4129,73.2500,30.244925,-1.239000e-06,-30.244926
6,U24174819,39698286581,9439183232,VOO,STK,USD,BUY,2026-05-05,0.6281,665.8700,418.232947,-1.884000e-06,-418.232949
7,U24174819,39698286587,9439183282,VOO,STK,USD,BUY,2026-05-05,1.0000,665.8649,665.864900,-3.000000e-06,-665.864903
8,U24174819,39698286588,9439183313,VOO,STK,USD,BUY,2026-05-05,1.0000,665.8700,665.870000,-3.000000e-06,-665.870003
9,U24174819,39698411555,9439215212,VXUS,STK,USD,BUY,2026-05-05,5.0000,83.5300,417.650000,-1.500000e-05,-417.650015


In [4]:
positions = ibkr.load_position_snapshots(config)
latest_positions = positions[positions["pulled_at"] == positions["pulled_at"].max()]
latest_positions

,pulled_at,account_id,symbol,asset_category,currency,report_date,quantity,mark_price,position_value
4,2026-07-01 16:38:26,U24174819,BND,STK,USD,2026-06-30,5.1238,73.41,376.14
5,2026-07-01 16:38:26,U24174819,QQQM,STK,USD,2026-06-30,6.1700,302.97,1869.32
6,2026-07-01 16:38:26,U24174819,VOO,STK,USD,2026-06-30,16.5760,686.81,11384.56
7,2026-07-01 16:38:26,U24174819,VXUS,STK,USD,2026-06-30,19.6643,85.49,1681.10
8,2026-07-01 16:38:26,U24174819,BND,STK,USD,2026-06-30,5.1238,73.41,376.14
9,2026-07-01 16:38:26,U24174819,QQQM,STK,USD,2026-06-30,6.1700,302.97,1869.32
10,2026-07-01 16:38:26,U24174819,VOO,STK,USD,2026-06-30,16.5760,686.81,11384.56
11,2026-07-01 16:38:26,U24174819,VXUS,STK,USD,2026-06-30,19.6643,85.49,1681.10
12,2026-07-01 16:38:26,U24174819,BND,STK,USD,2026-06-30,5.1238,73.41,376.14
13,2026-07-01 16:38:26,U24174819,QQQM,STK,USD,2026-06-30,6.1700,302.97,1869.32


In [5]:
cash = ibkr.load_cash_snapshots(config)
latest_cash = cash[cash["pulled_at"] == cash["pulled_at"].max()]
latest_cash

,pulled_at,account_id,currency,from_date,to_date,ending_cash,ending_settled_cash
2,2026-07-01 16:38:26,U24174819,BASE_SUMMARY,2025-07-01,2026-06-30,62.198884,2962.152075
3,2026-07-01 16:38:26,U24174819,USD,2025-07-01,2026-06-30,62.198884,2962.152075
4,2026-07-01 16:38:26,U24174819,BASE_SUMMARY,2025-07-01,2026-06-30,62.198884,2962.152075
5,2026-07-01 16:38:26,U24174819,USD,2025-07-01,2026-06-30,62.198884,2962.152075
6,2026-07-01 16:38:26,U24174819,BASE_SUMMARY,2025-07-01,2026-06-30,62.198884,2962.152075
7,2026-07-01 16:38:26,U24174819,USD,2025-07-01,2026-06-30,62.198884,2962.152075
8,2026-07-01 16:38:26,U24174819,BASE_SUMMARY,2025-07-01,2026-06-30,62.198884,2962.152075
9,2026-07-01 16:38:26,U24174819,USD,2025-07-01,2026-06-30,62.198884,2962.152075
